In [1]:
import os
from langgraph.graph import StateGraph,START,END,MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage,ToolMessage,AIMessage      
from langchain_openai import AzureChatOpenAI,AzureOpenAIEmbeddings
from dotenv import load_dotenv,find_dotenv
from IPython.display import Image

load_dotenv(find_dotenv(),override=True)
endpoint = os.getenv("ENDPOINT_URL")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version="2025-01-01-preview"

@tool
def currentTemperature(country:str) -> int :
    '''current temperature of the country'''
    if country.startswith("i"):
         return 10
    elif country.startswith("u"):
        return 20
    else:
        return 30

tools=[currentTemperature]
llm = (AzureChatOpenAI(azure_endpoint=endpoint,api_key=subscription_key,api_version=api_version)).bind_tools(tools)

def call_model(state:MessagesState):
    return {"messages": [ llm.invoke(state["messages"]) ]}

def should_continue(state:MessagesState):
    messages=state["messages"]
    last_message=messages[-1]
    if last_message.tool_calls:
        return "tools"
    else:
        return END

graph=StateGraph(MessagesState)
tool_node=ToolNode(tools)

graph.add_node("agent",call_model)
graph.add_node("tools",tool_node)

graph.add_edge(START,"agent")
graph.add_conditional_edges("agent",should_continue)
graph.add_edge("tools","agent")

app=graph.compile(checkpointer=MemorySaver())

In [3]:
res=app.invoke({"messages":[HumanMessage(content="what is the current temperature in india?")]}
                    ,config={"configurable":{"thread_id":"1"}})

